<a href="https://colab.research.google.com/github/ChrisCopeland123/Document_text_Analyzer/blob/main/Project_2_Milestone_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Manning Live Project

### Project 2: Smart Document Q&A System

### 1.1 Environment setup and test Embedding API

In [1]:
import os
import json
import numpy as np
import re
import openai
from typing import List, Dict, Any, Tuple
from google.colab import userdata

### 1.2 Document processor class


In [11]:
class DocumentProcessor:
    def __init__(self, model: str = "gpt-3.5-turbo"):
        """Initialize the DocumentProcessor with model configuration."""
        self.model = model
        self.documents: Dict[str, str] = {}

        # Initialize OpenAI client
        self.client = openai.OpenAI(api_key=userdata.get('OpenAI'))


    def chunk_text(self, text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
        """Split text into overlapping chunks for better context preservation."""
        if not text:
            return []

        # Split text into paragraphs first to avoid breaking in the middle of paragraphs
        paragraphs = re.split(r'\n\s*\n', text)

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            # If adding this paragraph exceeds chunk size and we already have content
            if len(current_chunk) + len(para) > chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                # Keep some overlap for context
                current_chunk = current_chunk[-overlap:] if overlap > 0 else ""

            # Add paragraph to current chunk
            current_chunk += para + "\n\n"

        # Add the last chunk if it has content
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        return chunks

    def _get_embedding(self, text: str) -> list:
        """Generate embedding for text using OpenAI's API."""
        if not text:
            return []

        try:
            response = self.client.embeddings.create(
                model="text-embedding-ada-002",
                input=text
            )
            return response.data[0].embedding

        except Exception as e:
            print(f"Error generating embedding: {e}")
            return []



    def add_document(self, doc_id: str, text: str) -> bool:
        """Process and add a document to the system."""
        if not text:
            print(f"Error: Empty text for document {doc_id}")
            return False

        # Store the original document
        self.documents[doc_id] = text

        # Chunk the document
        chunks = self.chunk_text(text)
        print(f"Document {doc_id} split into {len(chunks)} chunks")

        # Process each chunk to generate embeddings
        successful_chunks = 0
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i+1}/{len(chunks)}...")
            embedding = self._get_embedding(chunk)
            if embedding:
                successful_chunks += 1
            else:
                print(f"Failed to generate embedding for chunk {i+1}")

        print(f"Successfully processed {successful_chunks}/{len(chunks)} chunks")
        return successful_chunks > 0

    def add_document_from_file(self, file_path: str) -> bool:
        """Read and add a document from a file."""
        text = read_text_file(file_path)

        if text:
            doc_id = os.path.basename(file_path)
            return self.add_document(doc_id, text)
        return False

    def display_stats(self):
        """Display basic statistics about processed documents."""
        print(f"\nDocumentProcessor Statistics:")
        print(f"- Total documents: {len(self.documents)}")
        for doc_id, text in self.documents.items():
            chunks = self.chunk_text(text)
            print(f"- {doc_id}: {len(text)} characters, {len(chunks)} chunks")





In [5]:
# Helper function for reading files
def read_text_file(file_path: str) -> str:
    """Read content from a text file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return ""

### 1.3 Testing the document processor class

In [6]:
# Example usage and testing
def test_document_processor():
    """Test the DocumentProcessor with sample text."""

    # Create sample document
    sample_text = """
    Strategic Insights Partners Business Overview

    Strategic Insights Partners is a leading consulting firm specializing in data-driven
    business solutions. Founded in 2018 by Sarah Johnson and Michael Chen, the company
    has grown to serve over 200 clients across various industries.

    Our Services

    We provide comprehensive business analysis including market research, competitive
    analysis, and strategic planning. Our team of 45 analysts works with clients to
    identify growth opportunities and optimize operations.

    Recent Projects

    In 2023, we completed major projects for TechCorp Inc., Global Manufacturing Ltd.,
    and Regional Bank. These projects resulted in an average 15% improvement in
    operational efficiency for our clients.

    Technology and Innovation

    Our proprietary analytics platform processes over 10 million data points daily,
    enabling real-time insights and predictive modeling. We use advanced machine learning
    algorithms to identify patterns and trends that drive strategic decision-making.

    Future Growth

    Looking ahead to 2024, we plan to expand our services to include AI-powered document
    analysis and automated report generation. This expansion will allow us to serve
    clients more efficiently while maintaining our high standards of quality and accuracy.
    """

    # Save sample text to file
    with open("sample_business_doc.txt", "w", encoding="utf-8") as f:
        f.write(sample_text)

    print("Testing DocumentProcessor...")

    # Create DocumentProcessor instance
    processor = DocumentProcessor()

    # Test chunking first
    print("\n1. Testing text chunking:")
    chunks = processor.chunk_text(sample_text)
    print(f"Sample text split into {len(chunks)} chunks")

    for i, chunk in enumerate(chunks):
        print(f"\nChunk {i+1} ({len(chunk)} characters):")
        print("-" * 50)
        print(chunk[:100] + "..." if len(chunk) > 100 else chunk)

    # Test embedding generation
    print("\n2. Testing embedding generation:")
    test_chunk = chunks[0] if chunks else "Test text"
    embedding = processor._get_embedding(test_chunk)

    if embedding:
        print(f"Successfully generated embedding with {len(embedding)} dimensions")
        print(f"First 5 embedding values: {embedding[:5]}")
    else:
        print("Failed to generate embedding")

    # Test full document processing
    print("\n3. Testing full document processing:")
    success = processor.add_document_from_file("sample_business_doc.txt")

    if success:
        print("Document processing completed successfully!")
        processor.display_stats()
    else:
        print("Document processing failed!")

    print("\nTest completed!")

# Test different chunk sizes
def test_chunking_parameters():
    """Test different chunking parameters to understand their effects."""

    sample_text = """
    This is a test document with multiple paragraphs to demonstrate chunking behavior.

    Unit 7 woke up in a world of rust and silence.

    He found a faded photograph of a smiling child.

    With careful precision, he tried to mimic the expression on his metal face.

    A single spark flickered in his eyes as he finally understood love.

    Finally, this last paragraph discusses the importance of proper text segmentation
    for maintaining semantic coherence in retrieval-augmented generation systems.
    """

    processor = DocumentProcessor()

    print("Testing different chunking parameters:")
    print("=" * 60)

    # Test different chunk sizes
    for chunk_size in [200, 500, 1000]:
        for overlap in [50, 100, 200]:
            chunks = processor.chunk_text(sample_text, chunk_size=chunk_size, overlap=overlap)
            print(f"Chunk size: {chunk_size}, Overlap: {overlap} -> {len(chunks)} chunks")

            for i, chunk in enumerate(chunks):
                print(f"  Chunk {i+1}: {len(chunk)} characters")



In [12]:
# Run the tests
if __name__ == "__main__":
    test_document_processor()
    print("\n" + "="*60)
    test_chunking_parameters()

Testing DocumentProcessor...

1. Testing text chunking:
Sample text split into 2 chunks

Chunk 1 (801 characters):
--------------------------------------------------
Strategic Insights Partners Business Overview

    Strategic Insights Partners is a leading consulti...

Chunk 2 (742 characters):
--------------------------------------------------
r TechCorp Inc., Global Manufacturing Ltd.,
    and Regional Bank. These projects resulted in an ave...

2. Testing embedding generation:
Successfully generated embedding with 1536 dimensions
First 5 embedding values: [-0.014613453298807144, -0.0075482712127268314, -0.005344176199287176, -0.026596752926707268, -0.011701498180627823]

3. Testing full document processing:
Document sample_business_doc.txt split into 2 chunks
Processing chunk 1/2...
Processing chunk 2/2...
Successfully processed 2/2 chunks
Document processing completed successfully!

DocumentProcessor Statistics:
- Total documents: 1
- sample_business_doc.txt: 1355 characters, 2 ch